# Adversarial Threats to Cloud IDS — ROCm Pipeline v3
Native GPU adversarial training (Madry-style, no ART dependency for training). ART retained only for the paper-comparable fixed-budget FGSM/PGD/DeepFool evaluation track. Adaptive feature selection (cumulative importance, not fixed T). Per-layer configurable widths.


In [1]:
!pip install -q adversarial-robustness-toolbox shap imbalanced-learn tqdm pandas scikit-learn matplotlib



[notice] A new release of pip is available: 26.0.1 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# ===== SECTION 1: imports, seeds, CONFIG =====
import os
os.environ["HSA_OVERRIDE_GFX_VERSION"] = "11.0.0"  # must precede `import torch`

import glob, time, json, warnings
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import mutual_info_classif, VarianceThreshold
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from imblearn.over_sampling import SMOTE
import shap
from art.estimators.classification import PyTorchClassifier
from art.attacks.evasion import FastGradientMethod, ProjectedGradientDescent, DeepFool
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

DATA_SEED = int(os.environ.get("RUN_DATA_SEED", 42))
SEED = int(os.environ.get("RUN_SEED", 42))

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
try:
    torch.use_deterministic_algorithms(True, warn_only=True)
except Exception as e:
    print(f"determinism flag not fully supported: {e}")


class Logger:
    def __init__(self):
        self.t0 = time.time()
        self.n = 0

    def _t(self):
        s = time.time() - self.t0
        m, s = divmod(s, 60)
        h, m = divmod(m, 60)
        return f"{int(h):02d}:{int(m):02d}:{s:05.2f}"

    def section(self, title):
        self.n += 1
        print(f"\n{'='*78}\n>>> [{self._t()}] SECTION {self.n}: {title}\n{'='*78}")

    def info(self, msg): print(f"    . [{self._t()}] {msg}")
    def ok(self, msg): print(f"    v [{self._t()}] {msg}")
    def warn(self, msg): print(f"    ! [{self._t()}] WARNING: {msg}")
    def metric(self, label, value): print(f"    > [{self._t()}] {label:<42s} {value}")


log = Logger()

OUTPUT_DIR = f"./output/seed_{SEED}"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/checkpoints", exist_ok=True)

CONFIG = {
    # data
    "day_patterns": None,
    "max_rows_per_file": None,
    "test_size": 0.2,

    # feature selection - ADAPTIVE, not fixed T. Keep enough top-ranked features
    # to explain `*_cumulative_threshold` of total importance mass; count emerges
    # from the data instead of being asserted (paper leaves T unspecified anyway).
    "mi_cumulative_threshold": 0.90,
    "shap_cumulative_threshold": 0.90,
    "mi_var_thresh": 1e-5,
    "mi_sample_size": 50000,
    "shap_background_samples": 100,

    # architecture - each layer independently configurable
    "layer1_size": 128,
    "layer2_size": 64,
    "layer3_size": 32,
    "dropout": 0.3,
    "learning_rate": 0.001,
    "val_split": 0.1,
    "epochs_baseline": 50,
    "epochs_defended_stage1": 50,
    "epochs_defended_stage2": 15,
    "batch_size": 512,

    # fixed-budget evaluation attacks (paper-comparable track, via ART, CPU)
    "attack_eps": 0.05,
    "pgd_iters": 20,
    "pgd_eps_step_ratio": 0.1,
    "deepfool_iters": 50,
    "square_iters": 300,
    "square_p_init": 0.3,
    "n_attack_eval_samples": 2000,
    "n_square_samples": 1000,
    "n_shap_samples": 300,

    # native GPU PGD used for (a) minimal-perturbation eval, (b) Madry-style training.
    # Kept deliberately separate from the fixed-budget ART attacks above: this is the
    # "how hard can we make the attack / how small a perturbation still fools the
    # model" track, not the "reproduce the paper's exact eps" track.
    "minperturb_eps_grid": [0.005, 0.01, 0.02, 0.03, 0.05, 0.08, 0.10],
    "minperturb_alpha_ratio": 0.2,   # per-iter step size = eps * this ratio
    "minperturb_iters": 20,
    "minperturb_n_samples": 2000,

    # Madry-style adversarial training (Stage 2) - fresh PGD examples generated
    # from the model's CURRENT weights every batch, not a static pre-generated set.
    "adv_train_eps": 0.05,
    "adv_train_alpha_ratio": 0.25,
    "adv_train_pgd_iters": 7,        # fewer iters than eval PGD (20) for training speed
    "adv_train_ratio": 0.5,          # fraction of each batch replaced with adversarial examples
    "n_adv_train_samples": None,     # None = use full Stage-1 training set for Madry training

    "seed": SEED,
    "data_seed": DATA_SEED,
}

print(f"Run started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} | SEED={SEED} | DATA_SEED={DATA_SEED}")
log.info(f"OUTPUT_DIR={OUTPUT_DIR}")
log.info(f"CONFIG={json.dumps(CONFIG, indent=2)}")


def to_onehot(y, n_classes=2):
    return np.eye(n_classes, dtype=np.float32)[y]


# Paper's reported Table 4/5/6 figures (Holla, Polepalli & Sasikumar, IEEE Access 2025),
# hardcoded once here for the automated cross-check printed in Section 19.
PAPER_RESULTS = {
    "baseline_clean_acc": 85.0,
    "baseline_fgsm_acc": 60.0,
    "baseline_pgd_acc": 55.0,
    "baseline_deepfool_acc": 50.0,
    "baseline_square_acc": 63.0,
    "defended_clean_acc": None,   # not explicitly stated as a single number
    "defended_gradient_acc": 88.0,   # paper reports ~88% "restored" against gradient-based attacks
    "defended_square_acc": 84.0,
    "overhead_pct": 25.0,
}


Run started: 2026-08-03 01:30:12 | SEED=42 | DATA_SEED=42
    . [00:00:00.00] OUTPUT_DIR=./output/seed_42
    . [00:00:00.00] CONFIG={
  "day_patterns": null,
  "max_rows_per_file": null,
  "test_size": 0.2,
  "mi_cumulative_threshold": 0.9,
  "shap_cumulative_threshold": 0.9,
  "mi_var_thresh": 1e-05,
  "mi_sample_size": 50000,
  "shap_background_samples": 100,
  "layer1_size": 128,
  "layer2_size": 64,
  "layer3_size": 32,
  "dropout": 0.3,
  "learning_rate": 0.001,
  "val_split": 0.1,
  "epochs_baseline": 50,
  "epochs_defended_stage1": 50,
  "epochs_defended_stage2": 15,
  "batch_size": 512,
  "attack_eps": 0.05,
  "pgd_iters": 20,
  "pgd_eps_step_ratio": 0.1,
  "deepfool_iters": 50,
  "square_iters": 300,
  "square_p_init": 0.3,
  "n_attack_eval_samples": 2000,
  "n_square_samples": 1000,
  "n_shap_samples": 300,
  "minperturb_eps_grid": [
    0.005,
    0.01,
    0.02,
    0.03,
    0.05,
    0.08,
    0.1
  ],
  "minperturb_alpha_ratio": 0.2,
  "minperturb_iters": 20,
  "minpert

In [3]:
# ===== SECTION 2: device setup (ROCm) =====
log.section("Device setup")

CPU_DEVICE = torch.device("cpu")
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    log.ok(f"GPU acquired: {DEVICE} ({torch.cuda.get_device_name(0)})")
else:
    log.warn("CUDA/ROCm not available - falling back to CPU.")
    DEVICE = CPU_DEVICE

log.info(f"torch={torch.__version__} | train/infer/native-attack device={DEVICE} | ART fixed-budget attack device={CPU_DEVICE}")



>>> [00:00:00.17] SECTION 1: Device setup
    v [00:00:00.20] GPU acquired: cuda (AMD Radeon RX 7600)
    . [00:00:00.20] torch=2.9.1+rocm7.2.1 | train/infer/native-attack device=cuda | ART fixed-budget attack device=cpu


In [4]:
# ===== SECTION 3: data discovery + loading =====
log.section("Data discovery + loading")

DATA_DIR = "data"
if not os.path.isdir(DATA_DIR):
    raise FileNotFoundError(f"'{DATA_DIR}/' not found. Place the CSE-CIC-IDS2018 parquet/csv files there.")

all_files = sorted(glob.glob(os.path.join(DATA_DIR, "**", "*.parquet"), recursive=True))
if not all_files:
    all_files = sorted(glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True))
if not all_files:
    raise FileNotFoundError(f"No .parquet or .csv files found under '{DATA_DIR}/'.")
log.ok(f"Found {len(all_files)} file(s)")

DAY_PATTERNS = CONFIG["day_patterns"]
if DAY_PATTERNS is None:
    selected_files = all_files
else:
    selected_files = [p for p in all_files if any(pat in os.path.basename(p) for pat in DAY_PATTERNS)]
    if not selected_files:
        log.warn("No file matched day_patterns - using all found files.")
        selected_files = all_files

MAX_ROWS_PER_FILE = CONFIG["max_rows_per_file"]
dfs = []
for p in tqdm(selected_files, desc="Loading"):
    d = pd.read_parquet(p, engine="pyarrow") if p.endswith(".parquet") else pd.read_csv(p, low_memory=False)
    float_cols = d.select_dtypes(include=["float64"]).columns
    d[float_cols] = d[float_cols].astype(np.float32)
    if MAX_ROWS_PER_FILE is not None and len(d) > MAX_ROWS_PER_FILE:
        d = d.sample(n=MAX_ROWS_PER_FILE, random_state=DATA_SEED)
    log.info(f"{os.path.basename(p)}: {d.shape[0]:,} rows")
    dfs.append(d)

df = pd.concat(dfs, ignore_index=True)
del dfs
log.ok(f"Combined raw shape: {df.shape[0]:,} rows x {df.shape[1]} cols")



>>> [00:00:00.24] SECTION 2: Data discovery + loading
    v [00:00:00.24] Found 10 file(s)


Loading:   0%|          | 0/10 [00:00<?, ?it/s]

    . [00:00:00.44] Botnet-Friday-02-03-2018_TrafficForML_CICFlowMeter.parquet: 771,587 rows
    . [00:00:00.55] Bruteforce-Wednesday-14-02-2018_TrafficForML_CICFlowMeter.parquet: 619,346 rows
    . [00:00:00.69] DDoS1-Tuesday-20-02-2018_TrafficForML_CICFlowMeter.parquet: 954,846 rows
    . [00:00:00.77] DDoS2-Wednesday-21-02-2018_TrafficForML_CICFlowMeter.parquet: 561,396 rows
    . [00:00:00.92] DoS1-Thursday-15-02-2018_TrafficForML_CICFlowMeter.parquet: 794,812 rows
    . [00:00:01.01] DoS2-Friday-16-02-2018_TrafficForML_CICFlowMeter.parquet: 591,873 rows
    . [00:00:01.09] Infil1-Wednesday-28-02-2018_TrafficForML_CICFlowMeter.parquet: 456,873 rows
    . [00:00:01.13] Infil2-Thursday-01-03-2018_TrafficForML_CICFlowMeter.parquet: 249,170 rows
    . [00:00:01.27] Web1-Thursday-22-02-2018_TrafficForML_CICFlowMeter.parquet: 830,224 rows
    . [00:00:01.40] Web2-Friday-23-02-2018_TrafficForML_CICFlowMeter.parquet: 829,405 rows
    v [00:00:02.59] Combined raw shape: 6,659,532 rows x 78 

In [5]:
# ===== SECTION 4: data cleaning =====
log.section("Data cleaning")

df.columns = df.columns.str.strip()
n_before = df.shape[1]
df = df.loc[:, ~df.columns.duplicated()]
if df.shape[1] != n_before:
    log.warn(f"Dropped {n_before - df.shape[1]} duplicate-named column(s)")

cat_cols = df.select_dtypes(include="category").columns
if len(cat_cols) > 0:
    df[cat_cols] = df[cat_cols].astype(object)

label_col = next((c for c in df.columns if c.strip().lower() == "label"), None)
if label_col is None:
    raise KeyError(f"No 'Label' column found. Columns: {list(df.columns)}")
df = df.rename(columns={label_col: "Label"})

n_before = len(df)
df = df[df["Label"].astype(str).str.strip().str.lower() != "label"]
if n_before != len(df):
    log.warn(f"Dropped {n_before - len(df):,} embedded-header rows")

drop_cols = [c for c in ["Timestamp", "Flow ID", "Src IP", "Dst IP", "Source IP",
                          "Destination IP", "Src IP Addr", "Dst IP Addr"] if c in df.columns]
if drop_cols:
    df = df.drop(columns=drop_cols)

n_before = len(df)
df = df.drop_duplicates()
log.ok(f"Dropped {n_before - len(df):,} duplicate rows - {len(df):,} rows remain")



>>> [00:00:02.64] SECTION 3: Data cleaning
    v [00:00:44.36] Dropped 573,667 duplicate rows - 6,085,865 rows remain


In [6]:
# ===== SECTION 5: binary labeling + encoding =====
log.section("Binary labeling and encoding")

df["y"] = (df["Label"].astype(str).str.strip().str.lower() != "benign").astype(int)
df = df.drop(columns=["Label"])
log.ok(f"Binary label: {np.bincount(df['y'])} (0=benign, 1=attack)")

proto_col = next((c for c in df.columns if c.strip().lower() == "protocol"), None)
structural_features = []
if proto_col is not None:
    pre_cols = set(df.columns)
    df = pd.get_dummies(df, columns=[proto_col], prefix="proto")
    structural_features = sorted(set(df.columns) - pre_cols)
    log.info(f"One-hot encoded '{proto_col}' -> structural_features={structural_features}")
CONFIG["structural_features"] = structural_features

non_numeric = df.select_dtypes(include=["object", "category"]).columns
for c in non_numeric:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df = df.dropna()

X = df.drop(columns=["y"]).reset_index(drop=True).astype(np.float32)
y = df["y"].values.astype(np.int64)
log.ok(f"Final feature matrix: {X.shape} | class balance: {np.bincount(y)}")



>>> [00:00:44.41] SECTION 4: Binary labeling and encoding
    v [00:00:44.80] Binary label: [4760022 1325843] (0=benign, 1=attack)
    . [00:00:44.90] One-hot encoded 'Protocol' -> structural_features=['proto_0', 'proto_17', 'proto_6']
    v [00:00:45.65] Final feature matrix: (6085865, 79) | class balance: [4760022 1325843]


In [7]:
# ===== SECTION 6: adaptive mutual-information feature selection =====
log.section("Adaptive mutual-information feature selection")

def select_by_cumulative_importance(scores_series, threshold):
    """Keep the fewest top-ranked features whose cumulative share of total
    importance exceeds `threshold`. Feature COUNT emerges from the data instead
    of being asserted as a fixed T (the paper never specifies T either way)."""
    cumulative = scores_series.cumsum() / scores_series.sum()
    n_selected = int((cumulative < threshold).sum()) + 1
    return scores_series.head(n_selected).index.tolist()

def compute_mi_scores(X, y, var_thresh, mi_sample_size, seed):
    vt = VarianceThreshold(threshold=var_thresh)
    vt.fit(X)
    kept = X.columns[vt.get_support()]
    log.info(f"Variance threshold kept {len(kept)}/{X.shape[1]} features")
    X_v = X[kept]

    rng = np.random.RandomState(seed)
    if len(X_v) > mi_sample_size:
        idx = rng.choice(len(X_v), mi_sample_size, replace=False)
        X_mi, y_mi = X_v.iloc[idx], y[idx]
    else:
        X_mi, y_mi = X_v, y

    mi = mutual_info_classif(X_mi, y_mi, random_state=seed)
    return pd.Series(mi, index=X_v.columns).sort_values(ascending=False)

mi_scores = compute_mi_scores(X, y, CONFIG["mi_var_thresh"], CONFIG["mi_sample_size"], DATA_SEED)
selected_features = select_by_cumulative_importance(mi_scores, CONFIG["mi_cumulative_threshold"])
log.ok(f"Adaptively selected {len(selected_features)} features "
       f"(cumulative MI >= {CONFIG['mi_cumulative_threshold']*100:.0f}%): {selected_features}")



>>> [00:00:45.69] SECTION 5: Adaptive mutual-information feature selection
    . [00:00:51.81] Variance threshold kept 71/79 features
    v [00:01:04.23] Adaptively selected 42 features (cumulative MI >= 90%): ['Bwd Packets Length Total', 'Avg Bwd Segment Size', 'Subflow Bwd Bytes', 'Bwd Packet Length Mean', 'Fwd Packets Length Total', 'Subflow Fwd Bytes', 'Bwd Packet Length Max', 'Fwd Packet Length Max', 'Avg Fwd Segment Size', 'Fwd Packet Length Mean', 'Bwd Packet Length Std', 'Fwd Packet Length Std', 'Packet Length Variance', 'Packet Length Std', 'Packet Length Mean', 'Packet Length Max', 'Avg Packet Size', 'Fwd Header Length', 'Init Fwd Win Bytes', 'Fwd IAT Max', 'Fwd IAT Total', 'Init Bwd Win Bytes', 'Flow IAT Max', 'Bwd Header Length', 'Fwd IAT Mean', 'Flow Duration', 'Total Backward Packets', 'Subflow Bwd Packets', 'Flow IAT Mean', 'Fwd Packets/s', 'Fwd IAT Std', 'Flow IAT Std', 'Fwd IAT Min', 'Bwd IAT Min', 'Flow Packets/s', 'Total Fwd Packets', 'Bwd IAT Total', 'Subflow Fwd P

In [8]:
# ===== SECTION 7: split / scale / balance =====
log.section("Split / scale / balance")

X_sel = X[selected_features].values
X_train, X_test, y_train, y_test = train_test_split(
    X_sel, y, test_size=CONFIG["test_size"], random_state=DATA_SEED, stratify=y
)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).astype(np.float32)
X_test = scaler.transform(X_test).astype(np.float32)

sm = SMOTE(random_state=DATA_SEED)
X_train_bal, y_train_bal = sm.fit_resample(X_train, y_train)
X_train_bal = X_train_bal.astype(np.float32)
log.ok(f"Train={X_train.shape} Test={X_test.shape} SMOTE-balanced={X_train_bal.shape} balance={np.bincount(y_train_bal)}")



>>> [00:01:04.27] SECTION 6: Split / scale / balance
    v [00:16:44.36] Train=(4868692, 42) Test=(1217173, 42) SMOTE-balanced=(7616036, 42) balance=[3808018 3808018]


In [9]:
# ===== SECTION 8: model + core train/eval helpers =====

class IDSNet(nn.Module):
    def __init__(self, input_dim, layer_sizes=None, dropout=None):
        super().__init__()
        layer_sizes = layer_sizes or [CONFIG["layer1_size"], CONFIG["layer2_size"], CONFIG["layer3_size"]]
        dropout = CONFIG["dropout"] if dropout is None else dropout
        layers, prev = [], input_dim
        for h in layer_sizes:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 2))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


def train_model(model, X_train, y_train, epochs, batch_size, device,
                 desc="Training", val_split=None, seed=DATA_SEED, lr=None):
    val_split = CONFIG["val_split"] if val_split is None else val_split
    lr = CONFIG["learning_rate"] if lr is None else lr

    model.to(device)
    rng = np.random.RandomState(seed)
    idx = rng.permutation(len(X_train))
    n_val = max(1, int(len(X_train) * val_split))
    val_idx, tr_idx = idx[:n_val], idx[n_val:]

    train_ds = torch.utils.data.TensorDataset(
        torch.tensor(X_train[tr_idx], dtype=torch.float32),
        torch.tensor(y_train[tr_idx], dtype=torch.long),
    )
    train_loader = torch.utils.data.DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    X_val_t = torch.tensor(X_train[val_idx], dtype=torch.float32).to(device)
    y_val_t = torch.tensor(y_train[val_idx], dtype=torch.long).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    history = {"loss": [], "val_loss": [], "val_acc": []}
    pbar = tqdm(range(epochs), desc=desc)
    for _ in pbar:
        model.train()
        epoch_loss, n_batches = 0.0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            n_batches += 1
        model.eval()
        with torch.no_grad():
            val_out = model(X_val_t)
            val_loss = criterion(val_out, y_val_t).item()
            val_acc = (val_out.argmax(1) == y_val_t).float().mean().item()
        history["loss"].append(epoch_loss / max(n_batches, 1))
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        pbar.set_postfix(loss=f"{epoch_loss/max(n_batches,1):.4f}", val_acc=f"{val_acc:.4f}")
    return history


def predict_proba_numpy(model, X, device, batch_size=4096):
    model.eval()
    model.to(device)
    outs = []
    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            xb = torch.tensor(np.asarray(X[i:i + batch_size]), dtype=torch.float32).to(device)
            outs.append(torch.softmax(model(xb), dim=1).cpu().numpy())
    return np.vstack(outs)


def evaluate(model, X, y, device, label=""):
    t0 = time.time()
    probs = predict_proba_numpy(model, X, device)
    infer_ms = (time.time() - t0) / len(X) * 1000
    preds = probs.argmax(axis=1)
    m = dict(accuracy=accuracy_score(y, preds), f1=f1_score(y, preds, zero_division=0),
              precision=precision_score(y, preds, zero_division=0),
              recall=recall_score(y, preds, zero_division=0), infer_ms=infer_ms)
    if label:
        log.metric(label, f"Acc={m['accuracy']:.3f} F1={m['f1']:.3f} Prec={m['precision']:.3f} "
                           f"Rec={m['recall']:.3f} Infer={infer_ms:.4f}ms")
    return m


def build_mutable_mask(feature_names, structural_names):
    return np.array([f not in structural_names for f in feature_names])


def apply_feature_mask(X_adv, X_clean, mask):
    X_masked = X_clean.copy()
    X_masked[:, mask] = X_adv[:, mask]
    return X_masked

log.ok("Model + core helpers defined")


    v [00:16:44.43] Model + core helpers defined


In [10]:
# ===== SECTION 9: native GPU PGD + minimal-perturbation attack =====
# Pure PyTorch, runs entirely on DEVICE (ROCm) - no ART/CPU roundtrip. Used for
# (a) the minimal-perturbation evaluation below, and (b) Madry-style on-the-fly
# adversarial training in Stage 2. Ground-truth-targeted by construction (attack
# maximizes loss w.r.t. the TRUE label, not the model's own prediction).

def native_pgd(model, X, y, eps, alpha, iters, device, clip_min, clip_max,
               random_start=True, mask=None):
    """X, y: tensors already on `device`. Returns adversarial X (detached tensor).
    Temporarily switches the model to eval() during attack generation (frozen
    BatchNorm stats, disabled Dropout) so the attack gradient is not itself noisy
    from training-mode stochasticity - caller is responsible for restoring
    model.train() afterward if generating examples inside a training loop."""
    was_training = model.training
    model.eval()

    X_orig = X.detach().clone()
    X_adv = X.detach().clone()
    if random_start:
        noise = torch.empty_like(X_adv).uniform_(-eps, eps)
        if mask is not None:
            noise = noise * mask
        X_adv = torch.clamp(X_orig + noise, clip_min, clip_max)

    for _ in range(iters):
        X_adv.requires_grad_(True)
        loss = F.cross_entropy(model(X_adv), y)
        grad = torch.autograd.grad(loss, X_adv)[0]
        with torch.no_grad():
            step = alpha * grad.sign()
            if mask is not None:
                step = step * mask
            X_adv = X_adv + step
            delta = torch.clamp(X_adv - X_orig, -eps, eps)
            X_adv = torch.clamp(X_orig + delta, clip_min, clip_max)

    if was_training:
        model.train()
    return X_adv.detach()


def minimal_perturbation_sweep(model, X_np, y_np, eps_grid, alpha_ratio, iters, device,
                                clip_min, clip_max, mask=None, desc="Minimal-perturbation sweep"):
    """For each sample, find the SMALLEST eps in `eps_grid` (ascending) at which
    native_pgd successfully flips the prediction. This directly answers 'how small
    a perturbation is enough to fool the model' - the complement of the fixed-budget
    FGSM/PGD/DeepFool track, which only reports accuracy at one fixed eps.
    Returns an array of per-sample minimal eps (NaN = never fooled within eps_grid)."""
    X_t = torch.tensor(X_np, dtype=torch.float32).to(device)
    y_t = torch.tensor(y_np, dtype=torch.long).to(device)
    mask_t = torch.tensor(mask, dtype=torch.float32).to(device) if mask is not None else None

    n = len(X_np)
    min_eps_found = np.full(n, np.nan)
    still_unfooled = np.ones(n, dtype=bool)

    for eps in tqdm(eps_grid, desc=desc):
        if not still_unfooled.any():
            break
        alpha = eps * alpha_ratio
        idx = np.where(still_unfooled)[0]
        Xb, yb = X_t[idx], y_t[idx]
        X_adv = native_pgd(model, Xb, yb, eps, alpha, iters, device, clip_min, clip_max, mask=mask_t)
        with torch.no_grad():
            model.eval()
            preds = model(X_adv).argmax(1)
        fooled = (preds != yb).cpu().numpy()
        fooled_idx = idx[fooled]
        min_eps_found[fooled_idx] = eps
        still_unfooled[fooled_idx] = False

    return min_eps_found


def summarize_minperturb(min_eps_found, eps_grid):
    fooled_mask = ~np.isnan(min_eps_found)
    n_total = len(min_eps_found)
    n_fooled = int(fooled_mask.sum())
    summary = {
        "n_total": n_total,
        "n_fooled_within_grid": n_fooled,
        "pct_fooled_within_grid": 100.0 * n_fooled / max(n_total, 1),
        "eps_grid_max": max(eps_grid),
    }
    if n_fooled > 0:
        found = min_eps_found[fooled_mask]
        summary.update(mean_min_eps=float(found.mean()), median_min_eps=float(np.median(found)),
                        p10_min_eps=float(np.percentile(found, 10)), p90_min_eps=float(np.percentile(found, 90)))
    else:
        summary.update(mean_min_eps=None, median_min_eps=None, p10_min_eps=None, p90_min_eps=None)
    return summary

log.ok("Native PGD + minimal-perturbation functions defined")


    v [00:16:44.46] Native PGD + minimal-perturbation functions defined


In [11]:
# ===== SECTION 10: Madry-style on-the-fly adversarial training =====
# Standard robust-training recipe (Madry et al. 2018): every batch, generate FRESH
# adversarial examples using the model's CURRENT weights, then train on a mix of
# clean + adversarial examples. This is what actually builds robustness - a static
# pre-generated adversarial set (the earlier version of this pipeline) only teaches
# the model to resist attacks against an already-stale set of weights.

def adversarial_train_model(model, X_train, y_train, epochs, batch_size, device,
                             eps, alpha_ratio, pgd_iters, adv_ratio, clip_min, clip_max,
                             val_split=None, seed=DATA_SEED, lr=None, mask=None,
                             desc="Madry adversarial training"):
    val_split = CONFIG["val_split"] if val_split is None else val_split
    lr = CONFIG["learning_rate"] if lr is None else lr
    alpha = eps * alpha_ratio

    model.to(device)
    rng = np.random.RandomState(seed)
    idx = rng.permutation(len(X_train))
    n_val = max(1, int(len(X_train) * val_split))
    val_idx, tr_idx = idx[:n_val], idx[n_val:]

    train_ds = torch.utils.data.TensorDataset(
        torch.tensor(X_train[tr_idx], dtype=torch.float32),
        torch.tensor(y_train[tr_idx], dtype=torch.long),
    )
    train_loader = torch.utils.data.DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    X_val_t = torch.tensor(X_train[val_idx], dtype=torch.float32).to(device)
    y_val_t = torch.tensor(y_train[val_idx], dtype=torch.long).to(device)
    mask_t = torch.tensor(mask, dtype=torch.float32).to(device) if mask is not None else None

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    history = {"loss": [], "val_loss": [], "val_acc": [], "val_adv_acc": []}
    pbar = tqdm(range(epochs), desc=desc)
    for _ in pbar:
        model.train()
        epoch_loss, n_batches = 0.0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            n_adv = int(len(xb) * adv_ratio)

            if n_adv > 0:
                x_adv_part = native_pgd(model, xb[:n_adv], yb[:n_adv], eps, alpha, pgd_iters,
                                         device, clip_min, clip_max, mask=mask_t)
                xb_mixed = torch.cat([x_adv_part, xb[n_adv:]], dim=0)
            else:
                xb_mixed = xb

            model.train()
            optimizer.zero_grad()
            loss = criterion(model(xb_mixed), yb)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            n_batches += 1

        model.eval()
        with torch.no_grad():
            val_out = model(X_val_t)
            val_loss = criterion(val_out, y_val_t).item()
            val_acc = (val_out.argmax(1) == y_val_t).float().mean().item()
        val_adv = native_pgd(model, X_val_t, y_val_t, eps, alpha, pgd_iters, device, clip_min, clip_max, mask=mask_t)
        with torch.no_grad():
            model.eval()
            val_adv_acc = (model(val_adv).argmax(1) == y_val_t).float().mean().item()

        history["loss"].append(epoch_loss / max(n_batches, 1))
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        history["val_adv_acc"].append(val_adv_acc)
        pbar.set_postfix(loss=f"{epoch_loss/max(n_batches,1):.4f}", val_acc=f"{val_acc:.4f}", val_adv_acc=f"{val_adv_acc:.4f}")
    return history

log.ok("Madry-style adversarial training function defined")


    v [00:16:44.49] Madry-style adversarial training function defined


In [12]:
# ===== SECTION 11: ART helpers (paper-comparable fixed-budget track only) =====

def make_art_classifier(model, input_dim, clip_values):
    cpu_model = IDSNet(input_dim)
    cpu_model.load_state_dict(model.state_dict())
    cpu_model.to("cpu")
    cpu_model.eval()
    optimizer = torch.optim.Adam(cpu_model.parameters(), lr=CONFIG["learning_rate"])
    clf = PyTorchClassifier(model=cpu_model, loss=nn.CrossEntropyLoss(), optimizer=optimizer,
                             input_shape=(input_dim,), nb_classes=2, clip_values=clip_values, device_type="cpu")
    return clf, cpu_model


def cap_to_eps(X_adv, X_clean, eps, clip_values):
    delta = np.clip(X_adv - X_clean, -eps, eps)
    return np.clip(X_clean + delta, clip_values[0], clip_values[1]).astype(np.float32)


def make_art_attacks(clf, eps):
    return {
        "FGSM": FastGradientMethod(estimator=clf, eps=eps),
        "PGD": ProjectedGradientDescent(estimator=clf, eps=eps, eps_step=eps * CONFIG["pgd_eps_step_ratio"],
                                         max_iter=CONFIG["pgd_iters"]),
        "DeepFool": DeepFool(classifier=clf, max_iter=CONFIG["deepfool_iters"]),
    }


def attack_diagnostics(model, X_clean, X_adv, y_true, device):
    l_inf = np.abs(X_adv - X_clean).max(axis=1)
    l2 = np.linalg.norm(X_adv - X_clean, axis=1)
    clean_preds = predict_proba_numpy(model, X_clean, device).argmax(1)
    adv_preds = predict_proba_numpy(model, X_adv, device).argmax(1)
    was_correct = (clean_preds == y_true)
    n_correct = int(was_correct.sum())
    success_rate = float(((clean_preds != adv_preds) & was_correct).sum() / max(n_correct, 1))
    return dict(mean_linf=float(l_inf.mean()), max_linf=float(l_inf.max()), mean_l2=float(l2.mean()),
                success_rate=success_rate, n_originally_correct=n_correct)


def tabular_square_attack(model, X, y, device, eps, n_iter=None, p_init=None, seed=DATA_SEED, desc="Square Attack"):
    n_iter = CONFIG["square_iters"] if n_iter is None else n_iter
    p_init = CONFIG["square_p_init"] if p_init is None else p_init
    rng = np.random.RandomState(seed)
    n_samples, n_features = X.shape
    X_adv = X.copy()
    lo, hi = float(X.min()), float(X.max())

    def predict_labels(Xb):
        return predict_proba_numpy(model, Xb, device).argmax(axis=1)

    still_correct = predict_labels(X_adv) == y
    n_originally_correct = int(still_correct.sum())
    pbar = tqdm(range(n_iter), desc=desc)
    for it in pbar:
        n_remaining = int(still_correct.sum())
        pbar.set_postfix(remaining=n_remaining, total=n_samples)
        if n_remaining == 0:
            break
        p = max(p_init * (1 - it / n_iter), 0.05)
        n_perturb = max(1, int(p * n_features))
        active_idx = np.where(still_correct)[0]
        candidate = X_adv[active_idx].copy()
        for row in range(len(active_idx)):
            feat_idx = rng.choice(n_features, n_perturb, replace=False)
            delta = rng.choice([-eps, eps], size=n_perturb)
            candidate[row, feat_idx] = np.clip(candidate[row, feat_idx] + delta, lo, hi)
        preds = predict_labels(candidate)
        flipped = preds != y[active_idx]
        X_adv[active_idx[flipped]] = candidate[flipped]
        still_correct[active_idx[flipped]] = False
    pbar.close()

    success_rate = 1 - (int(still_correct.sum()) / max(n_originally_correct, 1))
    return X_adv, success_rate

log.ok("ART fixed-budget helpers defined")


    v [00:16:44.51] ART fixed-budget helpers defined


In [13]:
# ===== SECTION 12: baseline model training =====
log.section("Baseline model training")

input_dim = X_train_bal.shape[1]
baseline_model = IDSNet(input_dim)
log.info(f"input_dim={input_dim} layers=[{CONFIG['layer1_size']},{CONFIG['layer2_size']},{CONFIG['layer3_size']}] "
         f"epochs={CONFIG['epochs_baseline']} device={DEVICE}")

t0 = time.time()
train_model(baseline_model, X_train_bal, y_train_bal, epochs=CONFIG["epochs_baseline"],
            batch_size=CONFIG["batch_size"], device=DEVICE, desc="Baseline")
baseline_train_time = time.time() - t0
log.ok(f"Baseline trained in {baseline_train_time/60:.2f} min")

torch.save(baseline_model.state_dict(), f"{OUTPUT_DIR}/checkpoints/baseline_seed{SEED}.pt")
baseline_clean_metrics = evaluate(baseline_model, X_test, y_test, DEVICE, "Baseline (clean)")



>>> [00:16:44.54] SECTION 7: Baseline model training
    . [00:16:44.54] input_dim=42 layers=[128,64,32] epochs=50 device=cuda


Baseline:   0%|          | 0/50 [00:00<?, ?it/s]

    v [01:24:28.21] Baseline trained in 67.73 min
    > [01:24:29.03] Baseline (clean)                           Acc=0.976 F1=0.943 Prec=0.989 Rec=0.900 Infer=0.0004ms


In [14]:
# ===== SECTION 13: fixed-budget attacks vs baseline (paper-comparable, ART/CPU) =====
log.section("Fixed-budget attacks vs baseline")

EPS = CONFIG["attack_eps"]
clip_values = (float(X_train_bal.min()), float(X_train_bal.max()))
classifier, _ = make_art_classifier(baseline_model, input_dim, clip_values)
mutable_mask = build_mutable_mask(selected_features, CONFIG["structural_features"])
art_attacks = make_art_attacks(classifier, EPS)

N_ATTACK = min(CONFIG["n_attack_eval_samples"], len(X_test))
idx = np.random.RandomState(SEED).choice(len(X_test), N_ATTACK, replace=False)
X_atk, y_atk = X_test[idx], y_test[idx]

adv_results_before = {}
X_adv_store = {}
for name, atk in tqdm(list(art_attacks.items()), desc="Fixed-budget attacks vs baseline"):
    X_adv = atk.generate(x=X_atk, y=to_onehot(y_atk))
    X_adv = apply_feature_mask(X_adv, X_atk, mutable_mask)
    if name == "DeepFool":
        X_adv = cap_to_eps(X_adv, X_atk, EPS, clip_values)
    X_adv_store[name] = X_adv
    adv_results_before[name] = evaluate(baseline_model, X_adv, y_atk, DEVICE, f"Baseline under {name}")
    diag = attack_diagnostics(baseline_model, X_atk, X_adv, y_atk, DEVICE)
    adv_results_before[name].update(diag)



>>> [01:24:29.08] SECTION 8: Fixed-budget attacks vs baseline


Fixed-budget attacks vs baseline:   0%|          | 0/3 [00:00<?, ?it/s]

    > [01:24:29.39] Baseline under FGSM                        Acc=0.539 F1=0.163 Prec=0.133 Rec=0.213 Infer=0.0010ms


PGD - Batches:   0%|          | 0/63 [00:00<?, ?it/s]

    > [01:24:32.59] Baseline under PGD                         Acc=0.405 F1=0.036 Prec=0.027 Rec=0.052 Infer=0.0010ms


DeepFool:   0%|          | 0/2000 [00:00<?, ?it/s]

    > [01:24:41.64] Baseline under DeepFool                    Acc=0.163 F1=0.116 Prec=0.075 Rec=0.260 Infer=0.0010ms


In [15]:
# ===== SECTION 14: minimal-perturbation attack vs baseline (native GPU PGD) =====
log.section("Minimal-perturbation attack vs baseline")

N_MP = min(CONFIG["minperturb_n_samples"], len(X_test))
idx_mp = np.random.RandomState(SEED).choice(len(X_test), N_MP, replace=False)
X_mp, y_mp = X_test[idx_mp], y_test[idx_mp]

min_eps_baseline = minimal_perturbation_sweep(
    baseline_model, X_mp, y_mp, CONFIG["minperturb_eps_grid"], CONFIG["minperturb_alpha_ratio"],
    CONFIG["minperturb_iters"], DEVICE, clip_values[0], clip_values[1],
    mask=mutable_mask, desc="Min-perturbation vs baseline",
)
minperturb_summary_baseline = summarize_minperturb(min_eps_baseline, CONFIG["minperturb_eps_grid"])
log.ok(f"Baseline: {minperturb_summary_baseline['pct_fooled_within_grid']:.1f}% fooled within "
       f"eps<={minperturb_summary_baseline['eps_grid_max']}, mean min-eps="
       f"{minperturb_summary_baseline['mean_min_eps']}")



>>> [01:24:41.70] SECTION 9: Minimal-perturbation attack vs baseline


Min-perturbation vs baseline:   0%|          | 0/7 [00:00<?, ?it/s]

    v [01:24:42.38] Baseline: 95.8% fooled within eps<=0.1, mean min-eps=0.033073107049608357


In [16]:
# ===== SECTION 15: Square Attack vs baseline =====
log.section("Square Attack vs baseline")

N_SQ = min(CONFIG["n_square_samples"], len(X_test))
idx_sq = np.random.RandomState(SEED).choice(len(X_test), N_SQ, replace=False)
X_sq, y_sq = X_test[idx_sq], y_test[idx_sq]

X_adv_square, square_success_before = tabular_square_attack(baseline_model, X_sq, y_sq, DEVICE, EPS, desc="Square (baseline)")
square_before = evaluate(baseline_model, X_adv_square, y_sq, DEVICE, "Baseline under Square")
square_before["success_rate"] = square_success_before



>>> [01:24:42.44] SECTION 10: Square Attack vs baseline


Square (baseline):   0%|          | 0/300 [00:00<?, ?it/s]

    > [01:24:46.47] Baseline under Square                      Acc=0.242 F1=0.104 Prec=0.069 Rec=0.208 Infer=0.0015ms


In [17]:
# ===== SECTION 16: adaptive SHAP robust feature selection =====
log.section("Adaptive SHAP feature stability analysis")

N_SHAP = min(CONFIG["n_shap_samples"], len(X_atk))
N_SHAP_BG = min(CONFIG["shap_background_samples"], len(X_train_bal))
background = X_train_bal[np.random.RandomState(SEED).choice(len(X_train_bal), N_SHAP_BG, replace=False)]

def predict_fn(x):
    return predict_proba_numpy(baseline_model, x, DEVICE)

explainer = shap.Explainer(predict_fn, background)
shap_clean = explainer(X_atk[:N_SHAP])
shap_adv = explainer(X_adv_store["PGD"][:N_SHAP])

shap_clean_vals = np.abs(shap_clean.values[..., 1]).mean(axis=0)
shap_adv_vals = np.abs(shap_adv.values[..., 1]).mean(axis=0)
stability = 1 - np.abs(shap_clean_vals - shap_adv_vals) / (shap_clean_vals + 1e-9)
stability_series = pd.Series(stability, index=selected_features).sort_values(ascending=False)

robust_features = select_by_cumulative_importance(stability_series.clip(lower=0), CONFIG["shap_cumulative_threshold"])
log.ok(f"Adaptively selected {len(robust_features)} SHAP-robust features "
       f"(cumulative stability >= {CONFIG['shap_cumulative_threshold']*100:.0f}%): {robust_features}")



>>> [01:24:46.54] SECTION 11: Adaptive SHAP feature stability analysis


PermutationExplainer explainer: 301it [00:16, 10.81it/s]                         
PermutationExplainer explainer: 301it [00:12,  4.80it/s]                         

    v [01:25:16.09] Adaptively selected 24 SHAP-robust features (cumulative stability >= 90%): ['Bwd IAT Total', 'Avg Bwd Segment Size', 'Packet Length Std', 'Fwd Packet Length Max', 'Bwd Packet Length Mean', 'Bwd Packet Length Max', 'Packet Length Variance', 'Bwd IAT Max', 'Bwd IAT Std', 'Bwd Packet Length Std', 'Bwd IAT Min', 'Init Bwd Win Bytes', 'Bwd Packets/s', 'Fwd IAT Total', 'Avg Fwd Segment Size', 'Fwd Packet Length Mean', 'Packet Length Mean', 'Packet Length Max', 'Flow Duration', 'Fwd IAT Mean', 'Flow Packets/s', 'Init Fwd Win Bytes', 'Fwd Packet Length Std', 'Avg Packet Size']


In [18]:
# ===== SECTION 17: defended model - Stage 1 (clean, robust features) =====
log.section("Defended model - Stage 1")

feat_idx = [selected_features.index(f) for f in robust_features]
X_train_def = X_train_bal[:, feat_idx]
X_test_def = X_test[:, feat_idx]
X_atk_def = X_atk[:, feat_idx]
X_mp_def = X_mp[:, feat_idx]
mutable_mask_def = build_mutable_mask(robust_features, CONFIG["structural_features"])

defended_model = IDSNet(len(robust_features))
log.info(f"defended input_dim={len(robust_features)}")

t0 = time.time()
train_model(defended_model, X_train_def, y_train_bal, epochs=CONFIG["epochs_defended_stage1"],
            batch_size=CONFIG["batch_size"], device=DEVICE, desc="Stage 1")
t_stage1_time = time.time() - t0
log.ok(f"Stage 1 trained in {t_stage1_time/60:.2f} min")

torch.save(defended_model.state_dict(), f"{OUTPUT_DIR}/checkpoints/stage1_seed{SEED}.pt")



>>> [01:25:16.15] SECTION 12: Defended model - Stage 1
    . [01:25:17.21] defended input_dim=24


Stage 1:   0%|          | 0/50 [00:00<?, ?it/s]

    v [02:31:49.00] Stage 1 trained in 66.53 min


In [19]:
# ===== SECTION 18: defended model - Stage 2 (Madry-style adversarial training) =====
log.section("Defended model - Stage 2 (on-the-fly adversarial training)")

clip_values_def = (float(X_train_def.min()), float(X_train_def.max()))

t0 = time.time()
adv_train_history = adversarial_train_model(
    defended_model, X_train_def, y_train_bal,
    epochs=CONFIG["epochs_defended_stage2"], batch_size=CONFIG["batch_size"], device=DEVICE,
    eps=CONFIG["adv_train_eps"], alpha_ratio=CONFIG["adv_train_alpha_ratio"],
    pgd_iters=CONFIG["adv_train_pgd_iters"], adv_ratio=CONFIG["adv_train_ratio"],
    clip_min=clip_values_def[0], clip_max=clip_values_def[1], mask=mutable_mask_def,
    desc="Stage 2 (Madry)",
)
t_stage2_time = time.time() - t0
log.ok(f"Stage 2 trained in {t_stage2_time/60:.2f} min | "
       f"final val_acc={adv_train_history['val_acc'][-1]:.4f} val_adv_acc={adv_train_history['val_adv_acc'][-1]:.4f}")

t_gen_time = 0.0  # no separate static generation step anymore - generation happens inline every batch
defended_train_time_total = t_stage1_time + t_stage2_time

torch.save(defended_model.state_dict(), f"{OUTPUT_DIR}/checkpoints/defended_seed{SEED}.pt")



>>> [02:31:49.11] SECTION 13: Defended model - Stage 2 (on-the-fly adversarial training)


Stage 2 (Madry):   0%|          | 0/15 [00:00<?, ?it/s]

    v [03:17:44.12] Stage 2 trained in 45.92 min | final val_acc=0.9444 val_adv_acc=0.9166


In [20]:
# ===== SECTION 19: defended model evaluation =====
log.section("Defended model evaluation")

defended_clean_metrics = evaluate(defended_model, X_test_def, y_test, DEVICE, "Defended (clean)")

clip_values_def2 = (float(X_test_def.min()), float(X_test_def.max()))
classifier_def2, _ = make_art_classifier(defended_model, len(robust_features), clip_values_def2)
art_attacks_def = make_art_attacks(classifier_def2, EPS)

adv_results_after = {}
for name, atk in tqdm(list(art_attacks_def.items()), desc="Fixed-budget attacks vs defended"):
    X_adv2 = atk.generate(x=X_atk_def, y=to_onehot(y_atk))
    X_adv2 = apply_feature_mask(X_adv2, X_atk_def, mutable_mask_def)
    if name == "DeepFool":
        X_adv2 = cap_to_eps(X_adv2, X_atk_def, EPS, clip_values_def2)
    adv_results_after[name] = evaluate(defended_model, X_adv2, y_atk, DEVICE, f"Defended under {name}")
    diag = attack_diagnostics(defended_model, X_atk_def, X_adv2, y_atk, DEVICE)
    adv_results_after[name].update(diag)

min_eps_defended = minimal_perturbation_sweep(
    defended_model, X_mp_def, y_mp, CONFIG["minperturb_eps_grid"], CONFIG["minperturb_alpha_ratio"],
    CONFIG["minperturb_iters"], DEVICE, clip_values_def2[0], clip_values_def2[1],
    mask=mutable_mask_def, desc="Min-perturbation vs defended",
)
minperturb_summary_defended = summarize_minperturb(min_eps_defended, CONFIG["minperturb_eps_grid"])
log.ok(f"Defended: {minperturb_summary_defended['pct_fooled_within_grid']:.1f}% fooled within "
       f"eps<={minperturb_summary_defended['eps_grid_max']}, mean min-eps="
       f"{minperturb_summary_defended['mean_min_eps']}")

X_sq_def = X_sq[:, feat_idx]
X_adv_square2, square_success_after = tabular_square_attack(defended_model, X_sq_def, y_sq, DEVICE, EPS, desc="Square (defended)")
square_after = evaluate(defended_model, X_adv_square2, y_sq, DEVICE, "Defended under Square")
square_after["success_rate"] = square_success_after



>>> [03:17:48.87] SECTION 14: Defended model evaluation
    > [03:17:49.74] Defended (clean)                           Acc=0.960 F1=0.908 Prec=0.899 Rec=0.918 Infer=0.0004ms


Fixed-budget attacks vs defended:   0%|          | 0/3 [00:00<?, ?it/s]

    > [03:17:49.92] Defended under FGSM                        Acc=0.920 F1=0.830 Prec=0.757 Rec=0.920 Infer=0.0010ms


PGD - Batches:   0%|          | 0/63 [00:00<?, ?it/s]

    > [03:17:52.88] Defended under PGD                         Acc=0.913 F1=0.818 Prec=0.738 Rec=0.917 Infer=0.0023ms


...

    > [03:18:03.23] Defended under DeepFool                    Acc=0.938 F1=0.862 Prec=0.809 Rec=0.922 Infer=0.0015ms


Min-perturbation vs defended:   0%|          | 0/7 [00:00<?, ?it/s]

    v [03:18:03.47] Defended: 22.6% fooled within eps<=0.1, mean min-eps=0.06162251655629139


Square (defended):   0%|          | 0/300 [00:00<?, ?it/s]

    > [03:18:15.10] Defended under Square                      Acc=0.931 F1=0.851 Prec=0.785 Rec=0.929 Infer=0.0020ms


In [21]:
# ===== SECTION 20: summary tables =====
log.section("Summary tables")

def row(name, m, is_clean=False):
    sr = "-" if is_clean else f"{m.get('success_rate', float('nan'))*100:.2f}"
    return (name, round(m["accuracy"]*100, 2), round(m["f1"]*100, 2),
            round(m["precision"]*100, 2), round(m["recall"]*100, 2), sr)

rows = [row("Baseline (clean)", baseline_clean_metrics, True)]
for n in ["FGSM", "PGD", "DeepFool"]:
    rows.append(row(f"{n} (baseline)", adv_results_before[n]))
rows.append(row("Defended (clean)", defended_clean_metrics, True))
for n in ["FGSM", "PGD", "DeepFool"]:
    rows.append(row(f"{n} (defended)", adv_results_after[n]))

table4 = pd.DataFrame(rows, columns=["Condition", "Accuracy%", "F1%", "Precision%", "Recall%", "Success Rate%"])
print(table4.to_string(index=False))

table5 = pd.DataFrame([
    ("Baseline", square_before["accuracy"]*100, square_before["f1"]*100,
     square_before["precision"]*100, square_before["recall"]*100, square_before["success_rate"]*100),
    ("Defended", square_after["accuracy"]*100, square_after["f1"]*100,
     square_after["precision"]*100, square_after["recall"]*100, square_after["success_rate"]*100),
], columns=["Model", "Accuracy%", "F1%", "Precision%", "Recall%", "Success Rate%"]).round(2)
print()
print(table5.to_string(index=False))

table_minperturb = pd.DataFrame([
    {"Model": "Baseline", **minperturb_summary_baseline},
    {"Model": "Defended", **minperturb_summary_defended},
])
print()
print("Minimal-perturbation summary (how small an attack still fools the model):")
print(table_minperturb.to_string(index=False))

# FIX: overhead must be (defended - baseline) / baseline, not defended / baseline.
# The latter reports "defended time as a % of baseline time" (a ratio near/above
# 100%), not "extra time ON TOP OF baseline" (the actual overhead definition, and
# what the paper's ~25% figure represents).
overhead_retrain_pct = (t_stage2_time) / baseline_train_time * 100
overhead_full_pct = (defended_train_time_total - baseline_train_time) / baseline_train_time * 100

table6 = pd.DataFrame([
    ("Baseline", baseline_train_time/3600, baseline_clean_metrics["infer_ms"], "-", "-"),
    ("Defended", defended_train_time_total/3600, defended_clean_metrics["infer_ms"],
     f"{overhead_retrain_pct:.1f}%", f"{overhead_full_pct:.1f}%"),
], columns=["Model", "Train hrs", "Infer ms/sample", "Overhead(Stage2 only)", "Overhead(full pipeline)"])
print()
print(table6.to_string(index=False))



>>> [03:18:15.18] SECTION 15: Summary tables
          Condition  Accuracy%   F1%  Precision%  Recall% Success Rate%
   Baseline (clean)      97.61 94.25       98.91    90.02             -
    FGSM (baseline)      53.95 16.35       13.27    21.28         45.35
     PGD (baseline)      40.50  3.57        2.71     5.20         58.59
DeepFool (baseline)      16.25 11.61        7.47    26.00         85.53
   Defended (clean)      95.95 90.80       89.86    91.76             -
    FGSM (defended)      92.05 83.03       75.68    91.96          3.66
     PGD (defended)      91.35 81.77       73.76    91.73          4.35
DeepFool (defended)      93.75 86.19       80.91    92.20          2.15

   Model  Accuracy%  F1%  Precision%  Recall%  Success Rate%
Baseline       24.2 10.4        6.94    20.75          75.31
Defended       93.1 85.1       78.49    92.92           3.42

Minimal-perturbation summary (how small an attack still fools the model):
   Model  n_total  n_fooled_within_grid  pct_fo

In [22]:
# ===== SECTION 21: cross-check against paper's reported results =====
log.section("Cross-check vs paper (Holla, Polepalli & Sasikumar, IEEE Access 2025)")

# "gradient-based" comparison uses the mean of FGSM+PGD+DeepFool defended accuracy,
# matching how the paper reports a single ~88% figure across gradient attacks.
defended_gradient_mean = np.mean([adv_results_after[n]["accuracy"] for n in ["FGSM", "PGD", "DeepFool"]]) * 100

crosscheck = pd.DataFrame([
    ("Baseline clean accuracy", PAPER_RESULTS["baseline_clean_acc"], baseline_clean_metrics["accuracy"]*100),
    ("Baseline under FGSM", PAPER_RESULTS["baseline_fgsm_acc"], adv_results_before["FGSM"]["accuracy"]*100),
    ("Baseline under PGD", PAPER_RESULTS["baseline_pgd_acc"], adv_results_before["PGD"]["accuracy"]*100),
    ("Baseline under DeepFool", PAPER_RESULTS["baseline_deepfool_acc"], adv_results_before["DeepFool"]["accuracy"]*100),
    ("Baseline under Square", PAPER_RESULTS["baseline_square_acc"], square_before["accuracy"]*100),
    ("Defended under gradient attacks (mean)", PAPER_RESULTS["defended_gradient_acc"], defended_gradient_mean),
    ("Defended under Square", PAPER_RESULTS["defended_square_acc"], square_after["accuracy"]*100),
    ("Training overhead %", PAPER_RESULTS["overhead_pct"], overhead_full_pct),
], columns=["Metric", "Paper", "This run"])
crosscheck["Delta"] = (crosscheck["This run"] - crosscheck["Paper"]).round(2)
print(crosscheck.to_string(index=False))



>>> [03:18:15.22] SECTION 16: Cross-check vs paper (Holla, Polepalli & Sasikumar, IEEE Access 2025)
                                Metric  Paper  This run  Delta
               Baseline clean accuracy   85.0 97.608886  12.61
                   Baseline under FGSM   60.0 53.950000  -6.05
                    Baseline under PGD   55.0 40.500000 -14.50
               Baseline under DeepFool   50.0 16.250000 -33.75
                 Baseline under Square   63.0 24.200000 -38.80
Defended under gradient attacks (mean)   88.0 92.383333   4.38
                 Defended under Square   84.0 93.100000   9.10
                   Training overhead %   25.0 66.025720  41.03


In [23]:
# ===== SECTION 22: save all artifacts =====
log.section("Saving artifacts")

all_metrics = {
    "baseline_clean": baseline_clean_metrics,
    "baseline_under_attack": adv_results_before,
    "baseline_under_square": square_before,
    "baseline_minperturb": minperturb_summary_baseline,
    "defended_clean": defended_clean_metrics,
    "defended_under_attack": adv_results_after,
    "defended_under_square": square_after,
    "defended_minperturb": minperturb_summary_defended,
}
with open(f"{OUTPUT_DIR}/all_metrics.json", "w") as f:
    json.dump(all_metrics, f, indent=2)

run_config = dict(CONFIG)
run_config.update({
    "device": str(DEVICE),
    "n_attack_samples_used": N_ATTACK,
    "n_square_samples_used": N_SQ,
    "n_shap_samples_used": N_SHAP,
    "n_minperturb_samples_used": N_MP,
    "n_features_selected_mi": len(selected_features),
    "n_features_selected_shap": len(robust_features),
    "baseline_train_time_sec": baseline_train_time,
    "defended_stage1_time_sec": t_stage1_time,
    "defended_stage2_time_sec": t_stage2_time,
    "defended_total_time_sec": defended_train_time_total,
    "overhead_stage2_only_pct": overhead_retrain_pct,
    "overhead_full_pipeline_pct": overhead_full_pct,
})
with open(f"{OUTPUT_DIR}/run_config.json", "w") as f:
    json.dump(run_config, f, indent=2)

table4.to_csv(f"{OUTPUT_DIR}/table4.csv", index=False)
table5.to_csv(f"{OUTPUT_DIR}/table5.csv", index=False)
table6.to_csv(f"{OUTPUT_DIR}/table6.csv", index=False)
table_minperturb.to_csv(f"{OUTPUT_DIR}/table_minperturb.csv", index=False)
crosscheck.to_csv(f"{OUTPUT_DIR}/paper_crosscheck.csv", index=False)
mi_scores.to_csv(f"{OUTPUT_DIR}/mi_scores.csv")
stability_series.to_csv(f"{OUTPUT_DIR}/shap_stability_scores.csv")
with open(f"{OUTPUT_DIR}/selected_features.json", "w") as f:
    json.dump({"mi_features": selected_features, "shap_robust_features": robust_features}, f, indent=2)
torch.save(scaler, f"{OUTPUT_DIR}/scaler.pt")

log.ok(f"All artifacts saved to {OUTPUT_DIR}")



>>> [03:18:15.25] SECTION 17: Saving artifacts
    v [03:18:15.26] All artifacts saved to ./output/seed_42
